---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>
<h1 align="center">Course: Tools and Techniques for Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<h1 align="center">Lecture 6.8 (Data Preprocessing: Missing Values Imputation)</h1>

<a href="https://colab.research.google.com/github/arifpucit/data-science/blob/master/Section-4-Mathematics-for-Data-Science/Lec-4.1(Descriptive-Statistics).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <img align="center" width="900" src="images/ml-topimg1.png"  >

# Learning agenda of this notebook

- Overview of Data Pre-Processing and Feature Engineering
- Missing Data and its Types
    - MCAR
    - MAR
    - MNAR
- Univariate Imputation
    - Handling Missing Values using Panda's `fillna()` method
    - Handling Missing Values using sklearn's `SimpleImputer()` transformer
    - Use of Column Transformer
- Multivariate Imputation
    - Handling Missing Values using sklearn's `IterativeImputer()` transformer 
    - Handling Missing Values using sklearn's `KNNImputer()` transformer  
<br><br>
    
<img align="center" width="1000" src="images/imputer4.png"  >

# 1. Overview of  Data Pre-Processing and Feature Engineering
- `Data Preprocessing` involves actions that we need to perform on the dataset in order to make it ready to be fed to the machine learning model.
- `Feature Engineering` is the process of using domain knowledge to extract features from raw data via data mining techniques.<br><br>

<img align="center" width="800" src="images/housing-dataset3.png"  >
<br><br>

- Pre-processing package of sklearn provides a bundle of utility functions and transformer classes for data preprocessing (will cover later).
    - **Detecting and handling outliers** 
        - Univariate (Z-Score, IQR, Percentiles)
        - Multivariate Analysis (Depth-based, Distance-based, Density-based methods)
        - Trimming, Capping/Winsorization, Discritization
    - **Missing values Imputation** 
        - Univariate Imputation (Panda's `fillna()` method, Sklearn's `SimpleImputer()` transformer)
        - Multivariate Imputation (Sklearn's `IterativeImputer()` and  `KNNImputer()` transformers)
    - **Encoding Categorical Features**
        - Encode Nominal i/p features using Pandas `get_dummies()` and Scikit-Learn's `OneHotEncoder()`  
        - Encode Ordinal i/p features using Scikit-Learn's `OrdinalEncoder()`
        - Encode categorical o/p label using Scikit-Learn's `LabelEncoder()`
    - **Feature Scaling**
        - Use numPy to perform maxabs, minmax, standard and robust scaling
        - Use Sklearn's `MaxAbsScalar` , `MinMaxScalar`, `StandardScalar`, `RobustScalar` transformers
    - **Extracting Information** 
        - Use Sklearn's `CountVectorizer`, `DictVectorizer` , `TfidfVectorizer`, and `TfidfTransformer`
    - **Combining Information**
        - Use `FeatureUnion`, `Pipeline`, `PCA`

# 2. Missing Data and its Types
<center><h3 align="center"><div class="alert alert-success" style="margin: 20px">Missing data, or missing values, occur when no data value is stored for the variable in an observation .</h3></center>

[Inference and Missing Data, Donald Robin:](https://www.math.wsu.edu/faculty/xchen/stat115/lectureNotes3/Rubin%20Inference%20and%20Missing%20Data.pdf)  

    
- **Missing Completely At Random (MCAR):**
    - Missing data is INDEPENDENT of observed and unobserved variables in the dataset (No relationship).
    - In the case of MCAR, the data could be missing due to human error, some system/equipment failure, loss of sample, or some unsatisfactory technicalities while recording the values.
    - For example, incomplete filling of Google survey form due to Internet connection failure.
    - If your data is MCAR, the statistical analysis remains unbiased.
- **Missing At Random (MAR):**
    - Missing data is DEPENDENT on some observed variable(s) in the dataset (Some relationship).
    - For example, if you check the survey data, you may find that all the people have answered their ‘Gender’ but ‘Age’ values are mostly missing for people who have answered their ‘Gender’ as ‘female’.
    - If your data is MAR, the statistical analysis might result in bias. Getting an unbiased estimate of the parameters can be done only by modeling the missing data.  
- **Missing Not At Random (MNAR):**
   - Missing data is DEPENDENT on observed as well as unobserved variable(s) in the dataset.
   - For example, People having less income may refuse to share that information in a survey.
   - If your data is MNAR, the statistical analysis might result in bias. Getting an unbiased estimate of the parameters can be done only by modeling the missing data.

# 3. Techniques to Handle Missing Data 
- **Why do we need to handle missing values?** 
    - If the missing values are not handled properly, you may end up building a biased machine learning model which will lead to incorrect results.
    - Scikit-Learn's implementation of K-nearest and Naive Bayes do not support the presence of missing values.

- **How to treat missing values?**
    - Analyze each column with missing values carefully to understand the reasons behind the missing values as it is crucial to find out the strategy for handling the missing values. There are 2 primary ways of handling missing values:
        - **Delete the rows/columns having Missing values:** Drop rows (List-wise deletion) having missing values or drop the entire column, if missing data is MCAR and less than 5% data is missing.
        - **Impute the Missing Values:** Replace the missing value with a value
            - `Univariate Imputation` 
            - `Multivariate Imputation`
            
<img align="center" width="1000" src="images/imputer4.png"  >

# 4. Handling Missing Values using Pandas

## a. Load Dataset

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
df = pd.read_csv('datasets/loan-eligibility.csv')
df

## b. Identify Missing values

In [ ]:
# To check the count of non-null values in each column
df.info()

In [ ]:
# Returns a Boolean same sized object
df.isna()

In [ ]:
# Returns a series object showing count of missing values under each column
df.isna().sum()

In [ ]:
# List comprehension to get the list of column names having missing values
cols = [var for var in df.columns if df[var].isna().sum() > 0]
cols

In [ ]:
# Find the total number of missing values from the entire dataset
df.isna().sum().sum() 

## c. Deleting the Missing values
**Complete Case Analysis(CCA):** This is a quite straightforward method of handling the Missing Data, which directly removes the rows that have missing data i.e we consider only those rows where we have complete data i.e data is not missing. This method is also popularly known as “Listwise deletion”.

- **Assumptions:**
    - Data is Missing At Random(MAR).
    - Missing data is completely removed from the table.
- **Advantages:**
    - Easy to implement.
    - No Data manipulation required.
- **Limitations:**
    - Deleted data can be informative.
    - Can lead to the deletion of a large part of the data.
    - Can create a bias in the dataset, if a large amount of a particular type of variable is deleted from it.
    - The production model will not know what to do with Missing data.
- **When to Use:**
    - Data is MCAR (Missing Completely At Random) or MAR (Missing At Random).
    - Good for Mixed, Numerical, and Categorical data.
    - Missing data is not more than 5% – 6% of the dataset.
    - Data doesn’t contain much information and will not bias the dataset.

In [ ]:
df.isna().mean()*100

### (i) Deleting the entire row
- The `df.dropna()` method is used to drop the rows/columns having NaN values:

**`df.dropna(axis, how, subset, inplace)`**

- Where,
    - axis=0 is used to drop the row with `NaN` values (default)
    - axis=1 is used to drop the column with `NaN` values
    - how='any' is used to drop the row/column, if any single value in it is `NaN` (default)
    - how='all' is used to drop the row/column, if all values in it are `NaN`
    - inplace=False will return the new dataframe (default)
    - inplace=True will make change to original dataframe and returns None

In [ ]:
# You can use dropna() method to drop all the rows having NaN value
df1 = df.dropna(axis=0, how='any', inplace=False)
print(df1.shape)
df1.isnull().sum()

>- 134 rows out of 614 rows have been deleted, means you have deleted 22% of rows from your dataset :(

### (ii) Deleting the entire column
- If a certain column has lot of missing values then you can choose to drop the entire column.
- But this is an extreme case and should only be used when there are many null values in the column.



In [ ]:
# You can use dropna() method to drop all the rows having NaN value
df1 = df.dropna(axis=1, how='any', inplace=False)
print(df1.shape)
df1.isnull().sum()

>- 7 columns out of 12 input feature columns have been deleted, which is ofcourse not good :(

## d. Imputing the Missing Value (Univariate)

<center><h3 align="center"><div class="alert alert-success" style="margin: 20px">Imputation is a technique used for replacing the missing data with some substitute value to retain most of the information in the dataset</h3></center>

<img align="center" width="900" src="images/imputation1.png"  >


### Load Dataset

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
df = pd.read_csv('datasets/loan-eligibility.csv')
df.head()

In [ ]:
df.isnull().sum()

## e. Pandas `df.fillna()` Method for Imputation of Missing Values
- The `df.fillna()` method is used to fill NaN values using the specified `value` argument or the specified `method` argument.
- The only required argument is either the `value`, with which we want to replace the missing values OR the `method` to be used to replace the missing values

**`df.fillna(value, method, inplace)`**

- Where,
    - `value` argument specifies the value to be imputed at the place of missing values (required)
    - `method` can be either 
        - `ffill`, which means moves forward and fill NaN with previous value
        - `bfill`, which means moves backward and fill NaN with previous value
    - inplace=False will return the new dataframe with missing values filled (default)
    - inplace=True will make change to original dataframe with missing values filled and returns None

### (i) Replacing With Arbitrary Value
- If you can make an educated guess about the missing value then you can replace it with that value.
- This option is used when the data is not missing at random. If data is missing at random you should prefer mean/median

In [ ]:
# replace all NaNs under all the columns with a string value "missing"
df1 = df.fillna(value="missing", inplace=False)
df1.isnull().sum()

In [ ]:
#Replace the missing value under the Dependents column with '0'
df1 = df.copy()
df1['Dependents'] = df['Dependents'].fillna(value=0, inplace=False)
df1.isnull().sum()

### (ii) Replacing With Mean
- This is the most common method of imputing missing values of numeric columns. However, if there are outliers then the mean will not be appropriate. In such cases, outliers need to be treated first.
- While computing the mean the number of entities will be the number of non-null values

In [ ]:
df['LoanAmount'].mean()

In [ ]:
df['Credit_History'].mean()

In [ ]:
#Replace the missing values under the ‘LoanAmount’ and ‘Credit_History’ columns with mean value of those columns
df1['LoanAmount'] = df['LoanAmount'].fillna(value = df['LoanAmount'].mean())
df1['Credit_History'] = df['Credit_History'].fillna(value = df['Credit_History'].mean())
df1.isnull().sum()

### (iii) Replacing With Mode
- Mode is the most frequently occurring value. It is used in the case of categorical features.
- You can use the `fillna()` method for imputing the categorical columns ‘Gender’, ‘Married’, and ‘Self_Employed’.

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(nrows=1, ncols=3)
sns.countplot(x='Gender', data = df, ax=ax1)
sns.countplot(x='Married', data = df, ax=ax2)
sns.countplot(x='Self_Employed', data = df, ax=ax3)
plt.show();

In [ ]:
type(df.mode())

In [ ]:
#Replace the missing values for categorical columns with mode
#Replace the missing values under the ‘Gender’, ‘Married’, and ‘Self_Employed’ columns with mode
df1['Gender'] = df['Gender'].fillna(value = df['Gender'].mode()[0])
df1['Married'] = df['Married'].fillna(value = df['Married'].mode()[0])
df1['Self_Employed'] = df['Self_Employed'].fillna(value = df['Self_Employed'].mode()[0])
df1.isnull().sum()

### (iv) Replacing With Median
- Median is the middlemost value. It’s better to use the median value for imputation in the case of outliers.
- Moreover, we all know that if the distribution is skewed then the mean gets shifted, so median is a better candidate than mean.
- You can use ‘fillna’ method for imputing the column ‘Loan_Amount_Term’ with the median value.

In [ ]:
df['Loan_Amount_Term'].median()

In [ ]:
df1['Loan_Amount_Term']= df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].median())
df1.isnull().sum()

### (v) Replacing with previous value `ffill` or next value `bfill`
- In some cases, imputing the values with the previous value instead of mean, mode or median is more appropriate. This is called forward fill. It is mostly used in time series data.

In [ ]:
x = pd.Series(range(1,7))
x[2] = np.nan
x[4] = np.nan
x

In [ ]:
# Forward-Fill
x.fillna(method='ffill', inplace=False)

In [ ]:
# Backward-Fill
x.fillna(method='bfill', inplace=False)

## f. Check the impact of Imputation
- **Check out the following before after the imputation:**
    - Variance
    - Distribution
    - Outliers
    - Covariance and correlation with other columns

#### Check out change in `variance` of `LoanAmount` column, if we impute it with mean vs median

In [ ]:
df['LoanAmount_mean'] = df['LoanAmount'].fillna(df['LoanAmount'].mean())
df['LoanAmount_median'] = df['LoanAmount'].fillna(df['LoanAmount'].median())

print("Variance of original LoanAmount with missing values:", np.var(df['LoanAmount']))
print("Variance of imputed LoanAmount with mean:", np.var(df['LoanAmount_mean']))
print("Variance of imputed LoanAmount with median:", np.var(df['LoanAmount_median']))

#### Check out change in `distribution` of `LoanAmount` column, if we impute it with mean vs median

In [ ]:
df['LoanAmount_mean'] = df['LoanAmount'].fillna(df['LoanAmount'].mean())
df['LoanAmount_median'] = df['LoanAmount'].fillna(df['LoanAmount'].median())


fig, (ax1,ax2,ax3) = plt.subplots(nrows=1, ncols=3)
sns.histplot(df['LoanAmount'], kde=True, ax=ax1)
sns.histplot(df['LoanAmount_mean'], kde=True, ax=ax2)
sns.histplot(df['LoanAmount_median'], kde=True, ax=ax3)
plt.show();

# 5. Handling Missing Values using Scikit-Learn Transformers
[Data Transformers in Scikit-Learn:](https://scikit-learn.org/stable/data_transforms.html)    


<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Transformers are objects that transform a dataset in order to prepare it for predictive modeling. </h3>

    
## a. The `impute.SimpleImputer()` method of Scikit-Learn

#### Step 1: Import appropriate imputer class<br>
    
<center><font face = "Courier New" color=green size=5>from sklearn.</font><font face = "Courier New" color=blue size=5>impute</font> <font face = "Courier New" color=green size=5>  import </font><font face = "Courier New" color=magenta size=5> SimpleImputer,IterativeImputer,KNNImputer</font></center><br>

#### Step 2: Define imputer instance
<font face = "Courier New" color=blue size=4>imp = SimpleImputer(missing_values=np.nan, strategy='mean', fill_value=None)</font><br>
   
####  Step 3: Fit imputer instance on the dataset
<font face = "Courier New" color=blue size=4>imp.fit(X)</font><br>
    
    
####  Step 4: Transform the dataset
<font face = "Courier New" color=blue size=4>df = imp.transform(X)</font><br>


### Load Dataset

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
df = pd.read_csv('datasets/loan-eligibility.csv')
df.head()

In [ ]:
df.isnull().sum()

### Impute a constant value 0 for missing values under the `Dependents` Column

In [ ]:
from sklearn.impute import SimpleImputer
# define imputer
imp = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value=0)

In [ ]:
# fit on the dataset
imp.fit(df.iloc[:,3:4])

In [ ]:
# transform the dataset
df.iloc[:,3:4] = imp.transform(df.iloc[:,3:4])

In [ ]:
# verify
df.isnull().sum()

### Impute `most_frequent` value for missing values under the `Gender`,  `Married` and `Self-Employed` Column

In [ ]:
imp = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
df.iloc[:,[1,2,5]] = imp.fit_transform(df.iloc[:,[1,2,5]])

In [ ]:
df.isnull().sum()

### Impute `mean` value for missing values under the `LoanAmount`, `Loan_Amount_Term` and `Credit_History` Column

In [ ]:
imp = SimpleImputer(missing_values=np.nan, strategy='mean')
df.iloc[:,8:11] = imp.fit_transform(df.iloc[:,8:11])

In [ ]:
#verify
df.isnull().sum()

**Limitation of SimpleImputer**
- `SimpleImputer` is a transformer that works on entire data and it cannot be applied on a particular column. 
- For each missing value type, we have to define a separate imputer and fit-transform one by one. i.e., we have to create multiple instances of `SimpleImputer` class by specifying the appropriate strategy (mean, median,  most_frequent, constant) and then call `fit_transform()` by passing different columns that need to be transformed.
- Solution: `ColumnTransformer` that allows transformation in different columns with different imputations and applies at the same time.

## b. The `compose.ColumnTransformer()` method of Scikit-Learn

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">ColumnTransformer allows transformation in different columns with different imputations and applies at the same time </h3>

#### Step 1: Import ColumnTransformer<br>    
<center><font face = "Courier New" color=green size=5>from sklearn.</font><font face = "Courier New" color=blue size=5>compose</font> <font face = "Courier New" color=green size=5>  import </font><font face = "Courier New" color=magenta size=5> ColumnTransformer</font></center><br>


    
####  Step 2: Create SimpleImputer instances
<font face = "Courier New" color=blue size=4>imp_const = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value=0)</font><br>
<font face = "Courier New" color=blue size=4>imp_mode = SimpleImputer(missing_values=np.nan, strategy='most_frequent')</font><br>
<font face = "Courier New" color=blue size=4>imp_mean = SimpleImputer(missing_values=np.nan, strategy='mean')</font><br>
  
#### Step 3: Define ColumnTransformer instance
<font face = "Courier New" color=magenta size=4>column_trans = ColumnTransformer([<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;('impute_dep', imp_const, [3]),<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
                                 ('impute_gend-marr-emp', imp_mode, [1,2,5]),<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
                                 ('impute_loan-amt-cr', imp_mean, [8,9,10])<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
    ], <br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
                                 remainder='passthrough')
</font><br>
   
####  Step 4: Call fit_transform on ColumnTransformer instance
<font face = "Courier New" color=magenta size=4>result_arr = column_trans.fit_transform(df)</font><br>

### Load Dataset

In [ ]:
df = pd.read_csv('datasets/loan-eligibility.csv')
df.isnull().sum()

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# create the transformers
imp_const = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value=0)
imp_mode = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
imp_mean = SimpleImputer(missing_values=np.nan, strategy='mean')

# define ColumnTransformer 
column_trans = ColumnTransformer(
                                [('impute_dep', imp_const, [3]),
                                 ('impute_gend-marr-emp', imp_mode, [1,2,5]),
                                 ('impute_loan-amt-cr', imp_mean, [8,9,10])], 
                                 remainder='passthrough')

#Call fit-transform
result_arr = column_trans.fit_transform(df)
print(result_arr)
print(result_arr.shape)

In [ ]:
df_imputed = pd.DataFrame(data=result_arr, columns = df.columns)
df_imputed

In [ ]:
#verify
df_imputed.isnull().sum()

**Limitation of ColumnTransformer**
- In a `ColumnTransformer` we cannot apply multiple transforms to a single column. For example, if we have a categorical column and we want to apply first `SimpleImputer` and then `OneHotEncoder` to it, we cannot do this using `ColumnTransformer`.
- Solution: `Pipeline` which is a sequence of operations where output of one operation becomes input to its subsequent operation

## c. The Multivariate Imputation
<img align="center" width="1000" src="images/imputer4.png"  >

### (i) The `impute.IterativeImputer()` method of Scikit-Learn

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">IterativeImputer impute the missing values by modeling each feature having missing values as a function of other features in a round-robin fashion </h3>

**How the Algorithm Work?**
- Step 1: A feature column having NaN is designated as output and the other feature columns are treated as inputs. 
- Step 2: Regressor predicts missing output.
- Step 3: This is done for each feature in an iterative fashion, and then repeated for specified iterations.
- Step 4: Results from the final iteration are used for imputation
<br><br>
<img align="center" width="1000" src="images/imputer5.png"  >
<br><br>  
    
    
    
<font face = "Courier New" color=magenta size=4>it_imputer = IterativeImputer(estimator=None,<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;missing_values=np.nan,<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
                                 initial_strategy='mean',<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
                                 n_nearest_features=None,<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
    max_iter=10, <br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
    imputation_order='ascending')
</font><br>

### Sample Code

In [1]:
import pandas as pd
import numpy as np
X = pd.DataFrame({
                'x1':[np.nan,2,3,4,5],
                'x2':[6,np.nan,8,9,10],
                'x3':[11,12,np.nan,14,15],
                'x4':[16,17,18,19,np.nan]
                })
X

,x1,x2,x3,x4
0,NaN,6.0,11.0,16.0
1,2.0,NaN,12.0,17.0
2,3.0,8.0,NaN,18.0
3,4.0,9.0,14.0,19.0
4,5.0,10.0,15.0,NaN


> **Can you guess the missing values yourself in the above dataframe?**

In [2]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()

In [3]:
# Since IterativeImputer is still under experimentation, i.e., the API may change w/o any deprication sign
# therefore, we have to import enable_iterative_imputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
it_imp = IterativeImputer(estimator=lr, 
                          missing_values=np.nan, 
                          initial_strategy='mean', #mean, median, most_frequent, constant
                          max_iter=1,
                          n_nearest_features=None,
                         imputation_order='roman')# roman, arabic, random, ascending, descending
type(it_imp)

sklearn.impute._iterative.IterativeImputer

In [4]:
it_imp.fit_transform(X)

/opt/anaconda3/lib/python3.9/site-packages/sklearn/impute/_iterative.py:699: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


array([[ 1.,  6., 11., 16.],
       [ 2.,  7., 12., 17.],
       [ 3.,  8., 13., 18.],
       [ 4.,  9., 14., 19.],
       [ 5., 10., 15., 20.]])

### (ii) The `impute.KNNImputer()` method of Scikit-Learn

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Impute the missing values by using the mean value from K nearest neighbours found in the training set.
</h3>   

<center><font face = "Courier New" color=blue size=4>$\large d(p,q) = \sqrt{\sum_{i=1}^n{(q_i - p_i)^2}}$</font></center><br>
    
    
<center><font face = "Courier New" color=magenta size=4>$\large nan\_distance = \sqrt{weight * \text{(distance from present coordinates)}^2}$</font></center><br>
   
<font face = "Courier New" color=magenta size=4>$\hspace{11cm}weight = \large \frac{\text{Total number of coordinates}}{\text{number of present coordinates}}$</font><br>

- Nan-Euclidean distance calculates the Euclidean distance in the presence of missing values, by ignoring feature coordinates with a missing value in either sample and scales up the weight of the remaining coordinates.
<img align="center" width="700" src="images/knnworking.png"  >
<br><br>

- **Aglorithm to compute value in row 1, under column `x1`:**
    - Step 1: Compute the NaN-Distance between data point having missing value (row # 1) and every other datapoint (rows 0,2,3,4) in the dataset.
    - Step 2: Say `K=2`, so choose two data points having the minimum NaN-Distance. (Here the two nearest neighbours of row # 1 are row 0 and 2 (having minimum distance, i.e., 11.09 and 9)
    - Step 3: The value to be imputed is the mean of the `K=2` nearest neighbour's values under the column `x1`, i.e., 23 and 13.
      
    
<font face = "Courier New" color=magenta size=5>knn_im = KNNImputer(missing_values=np.nan,<br> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
                                 n_neighbors=5,<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
                                 weights='uniform',<br> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
    metric='nan_euclidean')
    </font>

**Example 1:**

In [5]:
import pandas as pd
X = pd.DataFrame({
                'x1':[23,np.nan, 13, 30,25],
                'x2':[np.nan, 35,41,np.nan,50],
                'x3':[57,58,61,71,69],
                'x4':[11,2,8,np.nan, np.nan]
                })
X

,x1,x2,x3,x4
0,23.0,NaN,57,11.0
1,NaN,35.0,58,2.0
2,13.0,41.0,61,8.0
3,30.0,NaN,71,NaN
4,25.0,50.0,69,NaN


In [6]:
from sklearn.impute import KNNImputer
knn_imp = KNNImputer(missing_values=np.nan,
                     n_neighbors=3,
                     weights='uniform', #Values of the neighbours are weighted equally.
                     metric='nan_euclidean')
type(knn_imp)

sklearn.impute._knn.KNNImputer

In [7]:
knn_imp.fit_transform(X)

array([[23., 42., 57., 11.],
       [22., 35., 58.,  2.],
       [13., 41., 61.,  8.],
       [30., 42., 71.,  7.],
       [25., 50., 69.,  7.]])

**Example 2:**

In [10]:
X = pd.DataFrame({
                'x1':[np.nan,2,3,4,5],
                'x2':[6,np.nan,8,9,10],
                'x3':[11,12,np.nan,14,15],
                'x4':[16,17,18,19,np.nan]
                })
X

,x1,x2,x3,x4
0,NaN,6.0,11.0,16.0
1,2.0,NaN,12.0,17.0
2,3.0,8.0,NaN,18.0
3,4.0,9.0,14.0,19.0
4,5.0,10.0,15.0,NaN


In [11]:
from sklearn.impute import KNNImputer
knn_imp = KNNImputer(missing_values=np.nan,
                     n_neighbors=1,
                     weights='uniform',
                     metric='nan_euclidean')
knn_imp.fit_transform(X)

array([[ 2.,  6., 11., 16.],
       [ 2.,  6., 12., 17.],
       [ 3.,  8., 12., 18.],
       [ 4.,  9., 14., 19.],
       [ 5., 10., 15., 19.]])

# Task To Do (Assignment)
### (i) Use `fillna()` and save the resulting dataframe
### (ii) Apply `SimpleImputer` using `ColumnTransformer` and save the resulting dataframe
### (iii) Apply `IterativeImputer` using `ColumnTransformer` and save the resulting dataframe
### (iv) Apply `KNNImputer` using `ColumnTransformer` and save the resulting dataframe
### (v) Compare the imputed values in all the resulting dataframes (Comment on the results)

In [9]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
df = pd.read_csv('datasets/loan-eligibility.csv')
df

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,LP002978,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural,Y
610,LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural,Y
611,LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban,Y
612,LP002984,Male,Yes,2,Graduate,No,7583,0.0,187.0,360.0,1.0,Urban,Y


In [52]:
X = pd.DataFrame({
                'x1':[3, np.nan, 5, 1, 8],
                'x2':[np.nan, 4, np.nan, 3, 6],
                'x3':[9, 4, np.nan, 7, 5],
                'x4':[6, 1, 5, 3, np.nan],
                'x5':[6, 9, 5, 3, np.nan]
                })
X

,x1,x2,x3,x4,x5
0,3.0,NaN,9.0,6.0,6.0
1,NaN,4.0,4.0,1.0,9.0
2,5.0,NaN,NaN,5.0,5.0
3,1.0,3.0,7.0,3.0,3.0
4,8.0,6.0,5.0,NaN,NaN


In [50]:
from sklearn.impute import KNNImputer
knn_imp = KNNImputer(missing_values=np.nan,
                     n_neighbors=2,
                     weights='uniform',
                     metric='nan_euclidean')
knn_imp.fit_transform(X)

array([[3. , 3.5, 9. , 6. , 6. ],
       [4.5, 4. , 4. , 1. , 9. ],
       [5. , 4.5, 8. , 5. , 5. ],
       [1. , 3. , 7. , 3. , 3. ],
       [8. , 6. , 5. , 3. , 7. ]])

In [46]:
np.sqrt(4*62/4)

7.874007874011811